In [1]:
import os
import cv2
import joblib
import numpy as np
import matplotlib.pyplot as plt

from tqdm import tqdm
from skimage.feature import hog
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

In [2]:
DATASET_PATH = r"C:\Users\ASUS\Documents\Projects\SignatureVerification\dataset\signatures"

In [3]:
def preprocess_image(image_path):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    # Resize to fixed size
    img = cv2.resize(img, (200,100))
    # Remove small noise
    img = cv2.GaussianBlur(img,(3,3),0)
    # Convert to binary
    _, img = cv2.threshold(
        img,
        0,
        255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    return img

In [4]:
def extract_hog(image):

    features = hog(
        image,
        orientations=9,
        pixels_per_cell=(8,8),
        cells_per_block=(2,2),
        visualize=False,
        block_norm="L2-Hys"
    )

    return features

In [5]:
def load_writer_data(writer_id):

    folder = os.path.join(
        DATASET_PATH,
        f"signatures_{writer_id}"
    )

    X = []
    y = []

    for file in sorted(os.listdir(folder)):

        path = os.path.join(folder,file)

        img = preprocess_image(path)

        feature = extract_hog(img)

        X.append(feature)

        if file.startswith("original"):
            y.append(1)

        elif file.startswith("forgeries"):
            y.append(0)

    return np.array(X), np.array(y)

In [6]:
X, y = load_writer_data(1)

print(X.shape)
print(y.shape)

print("Original :", np.sum(y==1))
print("Forgery :", np.sum(y==0))

(48, 9504)
(48,)
Original : 24
Forgery : 24


In [17]:
def train_writer_model(writer_id):

    print(f"\nTraining model for Writer {writer_id}...")

    # Load data
    X, y = load_writer_data(writer_id)

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42,
        stratify=y
    )

    # Scale features
    scaler = StandardScaler()

    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # Train model
    model = SVC(
        kernel="rbf",
        C=10,
        gamma="scale",
        probability=True,
        random_state=42
    )

    model.fit(X_train, y_train)

    # Prediction
    y_pred = model.predict(X_test)

    # Accuracy
    accuracy = accuracy_score(y_test, y_pred)

    print(f"Accuracy : {accuracy*100:.2f}%")

    # Classification Report
    print(classification_report(y_test, y_pred))

    # Save model
    BASE_DIR = os.getcwd()

    MODEL_DIR = os.path.join(BASE_DIR, "models")

    os.makedirs(MODEL_DIR, exist_ok=True)

    joblib.dump(
    model,
    os.path.join(
        MODEL_DIR,
        f"writer_{writer_id}.pkl"
    )
)

    joblib.dump(
        scaler,
        f"../models/writer_{writer_id}_scaler.pkl"
    )

    print("Model Saved Successfully!")

    return model, scaler

In [18]:
model, scaler = train_writer_model(1)


Training model for Writer 1...
Accuracy : 70.00%
              precision    recall  f1-score   support

           0       0.62      1.00      0.77         5
           1       1.00      0.40      0.57         5

    accuracy                           0.70        10
   macro avg       0.81      0.70      0.67        10
weighted avg       0.81      0.70      0.67        10

Model Saved Successfully!


In [19]:
import os

print(os.getcwd())

C:\Users\ASUS\Documents\Projects\SignatureVerification


In [20]:
print(os.path.abspath("../models"))

C:\Users\ASUS\Documents\Projects\models


In [21]:
print(os.path.abspath("../models"))

C:\Users\ASUS\Documents\Projects\models
